<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">

<br><br><br>
<h2><font color="#00586D" size=5>Capstone IX: Visualización y BI</font></h2>



<h1><font color="#00586D" size=6>Explorando datos de Airbnb</font></h1>

<br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Pedro Gómez López</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube V </font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>
<a id="inicio"></a>
   
    

</div>

---

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Presentación del caso de uso](#section1)
* [2. Generación de los conjuntos de datos](#section2)
* [3. Análisis inicial](#section3)
* [4. Análisis profundo](#section4)
* [5. Paneles e informes](#section5)
* [6. Casos de negocio](#section6)

---

<a id="section1"></a>
# <font color="#00586D"> 1. Presentación del caso de uso</font>

En este Capstone vamos a trabajar con datos de Airbnb. En concreto, vamos a analizar los datos de las propiedades de Airbnb en la ciudad de Valencia. El objetivo es explorar los datos y generar informes y paneles que nos permitan entender mejor el mercado de alquileres en esta ciudad. Podéis encontrar más información tanto en la página web de la que se obtienen los datos [enlace](https://insideairbnb.com/get-the-data/) como de la hoja de cálculo en la que vienen reflejados los respectivos data layouts [enlace](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?gid=1322284596#gid=1322284596).

Por lo tanto, lo primero será descargar los datos de `reviews` y `listings` de Valencia, en concreto los ficheros .csv.gz dentro de la sección `Get the Data` de la página [Get Inside Airbnb](https://insideairbnb.com/get-the-data/). Crearemos una carpeta `data` dentro de la raíz del proyecto y descargaremos los ficheros en esta carpeta.

---

<a id="section2"></a>
# <font color="#00586D"> 2. Generación de los conjuntos de datos</font>

Para la generación de los conjuntos de datos, utilizaremos la funcionalidad de [Supabase](https://supabase.com/) para la creación de una base de datos, a partir de la cual no solo tendremos acceso programático sino que también podremos realizar consultas SQL directamente. En este sentido, los pasos que tenemos que dar son los siguientes:
1. Crear una cuenta en Supabase.
2. Crear un nuevo proyecto.
3. Obtener los parámetros de acceso a la base de datos (modo Session pooler)

A continuación te facilito un código que nos permite conectarnos a la base de datos de Supabase y crear las tablas iniciales necesarias para comenzar a trabajar con el caso de uso. En este caso, vamos a crear dos tablas: una para los datos de las propiedades y otra para los datos de las reviews.

In [ ]:
import os
from dotenv import load_dotenv
import psycopg2
import pandas as pd
load_dotenv()  # Esto carga las variables del .env al entorno



def get_connection():
    return psycopg2.connect(
        host=os.environ.get("SUPABASE_HOST", "aws-0-eu-west-1.pooler.supabase.com"),
        port=5432,
        dbname=os.environ.get("SUPABASE_DB", "postgres"),
        user=os.environ.get("SUPABASE_USER", "postgres"),
        password=os.environ["SUPABASE_PASSWORD"]
    )

In [ ]:
try:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1;")
            result = cur.fetchone()
            print("Conexión exitosa. Resultado:", result)
except Exception as e:
    print("Error de conexión:", e)

In [ ]:
def create_table_from_df(sql):
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            conn.commit()

In [ ]:
def upload_table(df, sql, table_name, path):
    df.to_csv(path, index=False, quoting=1, na_rep="\\N")
    create_table_from_df(sql)
    with get_connection() as conn:
        with conn.cursor() as cur:
            with open(path, "r", encoding="utf-8") as f:
                cur.copy_expert(f'''
                    COPY {table_name} FROM STDIN WITH (FORMAT CSV, HEADER TRUE, QUOTE '"', NULL '\\N')
                ''', f)
            conn.commit()

In [ ]:
df_listings = pd.read_csv("./data/listings.csv")
apartamentos_create_table_sql = """
    CREATE TABLE IF NOT EXISTS apartamentos (
    id BIGINT,
    listing_url TEXT,
    scrape_id BIGINT,
    last_scraped TEXT,
    source TEXT,
    name TEXT,
    description TEXT,
    neighborhood_overview TEXT,
    picture_url TEXT,
    host_id BIGINT,
    host_url TEXT,
    host_name TEXT,
    host_since TEXT,
    host_location TEXT,
    host_about TEXT,
    host_response_time TEXT,
    host_response_rate TEXT,
    host_acceptance_rate TEXT,
    host_is_superhost TEXT,
    host_thumbnail_url TEXT,
    host_picture_url TEXT,
    host_neighbourhood TEXT,
    host_listings_count BIGINT,
    host_total_listings_count BIGINT,
    host_verifications TEXT,
    host_has_profile_pic TEXT,
    host_identity_verified TEXT,
    neighbourhood TEXT,
    neighbourhood_cleansed TEXT,
    neighbourhood_group_cleansed TEXT,
    latitude TEXT,
    longitude TEXT,
    property_type TEXT,
    room_type TEXT,
    accommodates BIGINT,
    bathrooms TEXT,
    bathrooms_text TEXT,
    bedrooms TEXT,
    beds TEXT,
    amenities TEXT,
    price TEXT,
    minimum_nights BIGINT,
    maximum_nights BIGINT,
    minimum_minimum_nights BIGINT,
    maximum_minimum_nights BIGINT,
    minimum_maximum_nights BIGINT,
    maximum_maximum_nights BIGINT,
    minimum_nights_avg_ntm TEXT,
    maximum_nights_avg_ntm TEXT,
    calendar_updated TEXT,
    has_availability TEXT,
    availability_30 BIGINT,
    availability_60 BIGINT,
    availability_90 BIGINT,
    availability_365 BIGINT,
    calendar_last_scraped TEXT,
    number_of_reviews BIGINT,
    number_of_reviews_ltm BIGINT,
    number_of_reviews_l30d BIGINT,
    availability_eoy BIGINT,
    number_of_reviews_ly BIGINT,
    estimated_occupancy_l365d BIGINT,
    estimated_revenue_l365d TEXT,
    first_review TEXT,
    last_review TEXT,
    review_scores_rating TEXT,
    review_scores_accuracy TEXT,
    review_scores_cleanliness TEXT,
    review_scores_checkin TEXT,
    review_scores_communication TEXT,
    review_scores_location TEXT,
    review_scores_value TEXT,
    license TEXT,
    instant_bookable TEXT,
    calculated_host_listings_count BIGINT,
    calculated_host_listings_count_entire_homes BIGINT,
    calculated_host_listings_count_private_rooms BIGINT,
    calculated_host_listings_count_shared_rooms BIGINT,
    reviews_per_month TEXT
    );
"""

In [ ]:
upload_table(df=df_listings, 
             sql=apartamentos_create_table_sql,
             table_name="apartamentos",
             path="./data/listings_airbnb.csv")

In [ ]:
df_reviews = pd.read_csv("./data/reviews.csv")
reviews_create_table_sql = """
    CREATE TABLE IF NOT EXISTS reviews (
    id BIGINT,
    listing_id BIGINT,
    date TEXT,
    reviewer_id BIGINT,
    reviewer_name TEXT,
    comments TEXT
    );
"""

In [11]:
df_reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,48154,117554,2010-10-12,180238,Martha,Toni's place was perfect in so many ways. It ...
1,48154,145645,2010-11-28,204240,Mark,Awesome stay!! We'd recommend Toni's apartment...
2,48154,190572,2011-03-01,258565,Domenico,really nice house in a wonderfull position! yo...
3,48154,195081,2011-03-08,213496,Romina & Martín,"Apartamento muy agradable, al igual que su pro..."
4,48154,218435,2011-04-05,340330,Jenna,"Was a great apartment, easy access to the site..."


In [ ]:
upload_table(df=df_reviews, 
             sql=reviews_create_table_sql,
             table_name="reviews",
             path="./data/reviews_airbnb.csv")

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicio 1</b></font><br>

Realiza una consulta a las dos nuevas tablas que has creado y comprueba que los datos se han insertado correctamente. Para ello obtén:
- El número de filas de cada tabla.
- El número de columnas de cada tabla.

Además, dentro de celda de comentario, explica qué información contiene cada tabla a alto nivel. 

In [ ]:
def check_table(table_name):
    with get_connection() as conn:
        with conn.cursor() as cur:
    
            cur.execute(f"SELECT COUNT(*) FROM {table_name};")
            row_count = cur.fetchone()[0]
            
   
            cur.execute(f"SELECT * FROM {table_name} LIMIT 1;")
            col_count = len(cur.description)
            
            print(f"Tabla '{table_name}': {row_count} filas, {col_count} columnas.")


check_table("apartamentos")
check_table("reviews")


<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>
La tabla apartamentos contiene información detallada sobre cada propiedad anunciada en Airbnb en Valencia, incluyendo datos de ubicación, características del alojamiento (capacidad, número de habitaciones, servicios), precios, condiciones de reserva y detalles del anfitrión. Cada fila representa un anuncio individual y permite analizar la oferta de alojamientos y sus características en el mercado local.

La tabla reviews almacena todas las reseñas de huéspedes sobre los alojamientos, incluyendo el ID de la propiedad, la fecha, el autor y el comentario de la review. Esta tabla permite analizar la reputación y satisfacción de los clientes para cada alojamiento, y puede relacionarse fácilmente con la tabla de apartamentos mediante el identificador de la propiedad.
</div>

---

<a id="section3"></a>
# <font color="#00586D"> 3. Análisis inicial</font>

Para comenzar a trabajar con los datos, los cargaremos dentro de nuestra plataforma de analítica y visualización online de datos, [Preset.io](https://preset.io/). Esta plataforma nos permite crear visualizaciones y paneles de control de forma sencilla y rápida, además de poder gestionar conexiones a diferentes bases de datos, crear datasets, charts y como no, paneles de control o dashboards. Para comenzar a trabajar con Preset tenemos que seguir los siguientes pasos:
1. Crear una cuenta en Preset, de tipo gratuito.
2. Crear un nuevo workspace, que llamaremos `CapstoneIX_username` donde `username` es vuestro nombre.
3. Dentro de ese workspace, tendréis que compartirlo con mi usuario para que tenga trazabilidad sobre el trabajo que habéis desarrollado.
4. Crear una nueva conexión a base de datos, que en este caso será de tipo `Other` y apuntará a nuestro entorno de Supabase, a la cual nos referiremos con el connection string en modo `session pooler`. Antes de añadir la fuente podremos testear la conexión para comprobar que funciona correctamente.


<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicio 2</b></font><br>

A continuación, puedes ir a la sección de `SQL Lab` dentro de Preset y ejecutar las mismas consultas que has realizado en el paso anterior para comprobar que los datos se han cargado correctamente.
- ¿Cuántas filas y columnas tiene cada tabla?

<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>
  Tabla 'apartamentos': 8847 filas, 79 columnas.  Tabla 'reviews': 416389 filas, 6 columnas.
</div>

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicio 3</b></font><br>

Parece que algunos tipos de datos tanto en la tabla de apartamentos como en la de reviews no son los correctos. Por ejemplo, el campo `price` debería ser un número y no un texto. Para ello, vamos a crear un nuevo dataset a partir de la tabla de apartamentos en el que las decisiones más importantes que tienes que tomar son:
- ¿Qué tipo de datos debería tener cada columna?
- ¿Qué columnas son necesarias y cuáles no?

Una vez tomadas estas decisiones, podrás modelar un nuevo dataset para cada conjunto de datos original de los que partimos. A continuación, adjunta la respuesta al ejercicio.

<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>

1. Query SQL para crear el nuevo dataset de apartamentos:

```sql
CREATE TABLE public.apartamentos_clean AS
SELECT
    id::BIGINT AS id,
    name::VARCHAR(255) AS name,
    neighbourhood_cleansed::VARCHAR(100) AS neighbourhood_cleansed,
    CAST(NULLIF(NULLIF(NULLIF(latitude, '\N'), ''), 'nan') AS FLOAT) AS latitude,
    CAST(NULLIF(NULLIF(NULLIF(longitude, '\N'), ''), 'nan') AS FLOAT) AS longitude,
    property_type::VARCHAR(100) AS property_type,
    room_type::VARCHAR(100) AS room_type,
    CAST(CAST(NULLIF(NULLIF(NULLIF(accommodates::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS accommodates,
    CAST(CAST(NULLIF(NULLIF(NULLIF(bedrooms, '\N'), ''), 'nan') AS FLOAT) AS INT) AS bedrooms,
    CAST(CAST(NULLIF(NULLIF(NULLIF(beds, '\N'), ''), 'nan') AS FLOAT) AS INT) AS beds,
    CASE
        WHEN price IN ('\N', '', 'nan') THEN NULL
        ELSE CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS NUMERIC(10,2))
    END AS price,
    CAST(CAST(NULLIF(NULLIF(NULLIF(minimum_nights::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS minimum_nights,
    CAST(CAST(NULLIF(NULLIF(NULLIF(maximum_nights::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS maximum_nights,
    CAST(CAST(NULLIF(NULLIF(NULLIF(number_of_reviews::TEXT, '\N'), ''), 'nan') AS FLOAT) AS INT) AS number_of_reviews,
    CAST(NULLIF(NULLIF(NULLIF(review_scores_rating, '\N'), ''), 'nan') AS FLOAT) AS review_scores_rating,
    host_id::BIGINT AS host_id,
    host_name::VARCHAR(255) AS host_name,
    CAST(NULLIF(NULLIF(NULLIF(host_since, '\N'), ''), 'nan') AS DATE) AS host_since,
    CASE
        WHEN host_is_superhost IN ('t', 'true', 'True', 'TRUE', 'yes', 'Yes', 'YES') THEN TRUE
        ELSE FALSE
    END AS host_is_superhost
FROM public.apartamentos;

```

2. Query SQL para crear el nuevo dataset de reviews:

```sql
CREATE TABLE public.reviews_clean AS
SELECT
    id::BIGINT AS id,
    listing_id::BIGINT AS listing_id,
    CAST(NULLIF(NULLIF(NULLIF(date, '\N'), ''), 'nan') AS DATE) AS date,
    reviewer_id::BIGINT AS reviewer_id,
    reviewer_name::VARCHAR(255) AS reviewer_name,
    comments::TEXT AS comments
FROM public.reviews;

```

3. Comentarios sobre el nuevo dataset de apartamentos  
Las columnas que he decidido mantener en el dataset de apartamentos son aquellas que recogen la información más relevante para el análisis de la oferta de alojamiento, como id, name, neighbourhood_cleansed, latitude, longitude, property_type, room_type, accommodates, bedrooms, beds, price, minimum_nights, maximum_nights, number_of_reviews, review_scores_rating, host_id, host_name, host_since y host_is_superhost. Estas columnas permiten estudiar la localización, características, capacidad, precios y reputación de cada alojamiento.  Por el contrario, he eliminado numerosas columnas que resultaban redundantes, excesivamente detalladas o poco relevantes para el análisis, tales como descripciones extensas (description, neighborhood_overview), URLs y rutas de imágenes, detalles avanzados sobre el anfitrión, información sobre verificación o respuesta, campos relacionados con calendarios, disponibilidad granular, métricas históricas poco significativas y otros metadatos generados automáticamente. Esta selección permite simplificar el modelo de datos y centrar el análisis en los factores clave del mercado de alquiler en Valencia.


4. Comentarios sobre el nuevo dataset de reviews  
Las columnas que he decidido mantener en el dataset de reviews son id, listing_id, date, reviewer_id, reviewer_name y comments, porque contienen la información esencial para analizar las reseñas de los usuarios, permitiendo estudiar la relación entre los comentarios y los alojamientos, así como evaluar la experiencia de los huéspedes a lo largo del tiempo.
No se han eliminado columnas.
</div>

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicio 4</b></font><br>

Ahora que tenemos los datasets creados y más limpios, podemos comenzar a trabajar con ellos. A continuación, vamos a lanzar algunas consultas SQL para obtener información básica de los datos. En concreto, queremos responder a las preguntas de negocio que se plantean a continuación:

- ¿Dónde están los alojamientos más caros según el tipo de alojamiento?
- ¿Cómo ha evolucionado la cantidad de reseñas a lo largo del tiempo? Hay algo que te llame la atención? Cómo lo explicarías?
- Cuáles son los mejores y peores barrios de Valencia según al relación calidad-precio?
- En qué barrios se distribuyen los grandes tenedores de propiedades? 
- Qué barrios han crecido más en volumen de propiedades en los últimos 3 años? Y de reviews?

Además, para cada una de las consultas SQL que has lanzado, añade un comentario explicando qué gráfico utilizarías para visualizar la información y por qué. Dentro de las consultas SQL, puedes crear directamente los gráficos que consideres necesarios para responder a las preguntas de negocio.

Las columnas 'listing_id' e'id' se han intercambiado en algun momento en el proceso de load a postgre, creo que ha sido porque al crear la tabla el orden se han alterado. Simplemente he permutado las columnas en la tabla final de postgre.


<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>

**1. Query SQL para obtener los alojamientos más caros según el tipo de alojamiento:**
```sql
SELECT DISTINCT ON (property_type)
    property_type,
    neighbourhood_cleansed,
    price
FROM public.apartamentos_clean
WHERE price IS NOT NULL
ORDER BY property_type, price DESC;
```
Utilizaría un gráfico de barras horizontal (bar chart) para visualizar la propiedad más cara por tipo de alojamiento, mostrando en el eje Y los distintos tipos de propiedad y en el eje X el precio máximo correspondiente. Este tipo de gráfico permite comparar de forma clara y rápida los precios máximos entre diferentes categorías de alojamiento. Alternativamente, si se quiere destacar la localización de estas propiedades más caras, se podría emplear un mapa de puntos donde cada propiedad se sitúe en función de su latitud y longitud, facilitando la identificación de las zonas donde se concentran los precios más altos.







**2. Query SQL para obtener la evolución de la cantidad de reseñas a lo largo del tiempo:**
```sql
SELECT 
    DATE_TRUNC('month', date) AS mes,
    COUNT(*) AS num_reviews
FROM public.reviews_clean
WHERE date IS NOT NULL
GROUP BY mes
ORDER BY mes;

```

Aquí usaría un gráfico de líneas temporal para mostrar la evolución mensual del número de reseñas. Así se identifican tendencias, estacionalidad, caídas o picos (como los causados por la pandemia, estacionalidad turística, etc).

Los resultados me llaman la atención porque reflejan claramente cómo la actividad del mercado de alquiler turístico en Valencia está muy influenciada por factores externos, como la pandemia de COVID-19, que provocó una caída abrupta en el número de reseñas en 2020, seguida de una recuperación rápida y un crecimiento aún mayor en los años posteriores. Además, los picos y caídas más recientes podrían indicar cambios en la demanda, en la oferta de alojamientos o simplemente que los datos del periodo más reciente aún no están completos, lo que evidencia la importancia de considerar el contexto temporal y la calidad de los datos al analizar tendencias.

**3. Query SQL para obtener los mejores y peores barrios de Valencia según la relación calidad-precio:**
```sql
SELECT 
    neighbourhood_cleansed,
    AVG(price) AS avg_price,
    AVG(review_scores_rating) AS avg_rating,
    (AVG(review_scores_rating) / NULLIF(AVG(price), 0)) AS value_for_money,
    COUNT(*) AS num_listings
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND review_scores_rating IS NOT NULL
GROUP BY neighbourhood_cleansed
HAVING COUNT(*) > 10  
ORDER BY value_for_money DESC;

```
Un gráfico de barras ordenado (horizontal) mostrando el ratio “calidad/precio” por barrio es ideal para comparar rápidamente porque permite ver claramente los barrios con mejor y peor ratio de manera rapida.

**4. Query SQL para obtener la distribución de los grandes tenedores de propiedades:**
```sql
WITH grandes_tenedores AS (
    SELECT host_id, COUNT(*) AS num_props
    FROM public.apartamentos_clean
    GROUP BY host_id
    HAVING COUNT(*) > 10  --Como no se especica que se considera gran tenedor, he considerado que +10 
)
SELECT 
    a.neighbourhood_cleansed,
    COUNT(DISTINCT a.host_id) AS num_grandes_tenedores,
    SUM(CASE WHEN g.host_id IS NOT NULL THEN 1 ELSE 0 END) AS num_propiedades_grandes_tenedores
FROM public.apartamentos_clean a
LEFT JOIN grandes_tenedores g ON a.host_id = g.host_id
GROUP BY a.neighbourhood_cleansed
ORDER BY num_propiedades_grandes_tenedores DESC;

```
El gráfico que he usado para visualizar esta información es un gráfico de barras horizontal, porque permite comparar de manera clara y rápida el número de grandes tenedores de propiedades en cada barrio, facilitando la identificación de aquellas zonas donde su concentración es mayor y ayudando a destacar visualmente las diferencias entre barrios.

**5. Query SQL para obtener los barrios que han crecido más en volumen de propiedades en los últimos 3 años:**
```sql
SELECT
    neighbourhood_cleansed,
    EXTRACT(YEAR FROM host_since) AS year,
    COUNT(*) AS num_new_props
FROM public.apartamentos_clean
WHERE host_since >= (CURRENT_DATE - INTERVAL '3 year')
GROUP BY neighbourhood_cleansed, year
ORDER BY num_new_props DESC;

```
He usado un gráfico de barras apiladas (por barrio y año) para mostrar el crecimiento en número de propiedades por barrio en los últimos años.
</div>

---

<a id="section4"></a>
# <font color="#00586D"> 4. Análisis profundo</font>

Una vez hemos comenzado a ver la vista preliminar de los datos, vamos a profundizar un poco más en el análisis. Para ello, vamos a intentar responder a las siguientes preguntas de negocio:

- ¿Cuál ha sido la evolución mensual de los ingresos por barrio?
- Qué factores crees que tienen un mayor impacto en el precio de los apartamentos? ¿Por qué? Valida tus intuiciones con queries SQL centradas en estudiar la relación entre el precio y los factores que has identificado.
- ¿Qué barrio ha tenido el mayor crecimiento interanual en número de reseñas durante los meses de Fallas y con un rating de 4 estrellas o más contando con más 30 reseñas?

Además, para cada una de las consultas SQL que has lanzado, añade un comentario explicando qué gráfico utilizarías para visualizar la información y por qué. Dentro de las consultas SQL, puedes crear directamente los gráficos que consideres necesarios para responder a las preguntas de negocio.

<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>

**1. Query SQL para obtener la evolución mensual de los ingresos por barrio:**
```sql
SELECT
    neighbourhood_cleansed,
    DATE_TRUNC('month', r.date) AS mes,
    SUM(a.price) AS ingresos_estimados
FROM public.reviews_clean r
JOIN public.apartamentos_clean a ON r.listing_id = a.id
WHERE r.date IS NOT NULL AND a.price IS NOT NULL
GROUP BY neighbourhood_cleansed, mes
ORDER BY mes, neighbourhood_cleansed;

```
He usado un gráfico de líneas múltiple (líneas separadas por barrio) para visualizar la evolución mensual de los ingresos estimados en cada barrio. Esto permite ver tendencias generales y también comparar la importancia relativa de cada barrio a lo largo del tiempo.

**2. Query SQL para obtener los factores que tienen un mayor impacto en el precio de los apartamentos:**

```sql
-- Relación entre precio y número de habitaciones
SELECT
    bedrooms,
    AVG(price) AS avg_price,
    COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND bedrooms IS NOT NULL
GROUP BY bedrooms
ORDER BY bedrooms;

--Relación entre precio y capacidad (accommodates)
SELECT
    accommodates,
    AVG(price) AS avg_price,
    COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND accommodates IS NOT NULL
GROUP BY accommodates
ORDER BY accommodates;

--Relación entre precio y rating medio
SELECT
    ROUND(review_scores_rating::numeric, 1) AS rating_rounded,
    AVG(price) AS avg_price,
    COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL AND review_scores_rating IS NOT NULL
GROUP BY rating_rounded
ORDER BY rating_rounded;

-- Relación entre precio y si el host es Superhost
SELECT
    host_is_superhost,
    AVG(price) AS avg_price,
    COUNT(*) AS num_apartments
FROM public.apartamentos_clean
WHERE price IS NOT NULL
GROUP BY host_is_superhost;


```
He utilizado gráficos de líneas y de barras para visualizar la relación entre distintos factores y el precio medio de los apartamentos. Los gráficos de líneas permiten ver cómo varía el precio en función del número de dormitorios, la capacidad máxima y la valoración media, mientras que el gráfico de barras compara directamente el precio medio entre apartamentos gestionados por superhosts y el resto de anfitriones.

A partir de estos análisis, se observa que el tamaño y la capacidad del apartamento influyen positivamente en el precio, mientras que, de forma llamativa, los apartamentos gestionados por superhosts presentan un precio medio inferior al de los anfitriones estándar. Esto puede deberse a que los superhosts, aunque ofrecen una experiencia y un servicio más valorados, tienden a buscar una mayor ocupación y mejores valoraciones ajustando sus precios, o bien gestionan propiedades más pequeñas o económicas.

**3. Query SQL para obtener el barrio que ha tenido el mayor crecimiento interanual en número de reseñas durante los meses de Fallas y con un rating de 4 estrellas o más contando con más 30 reseñas:**
```sql
WITH fallas_reviews AS (
    SELECT
        a.neighbourhood_cleansed AS barrio,
        EXTRACT(YEAR FROM r.date) AS anio,
        COUNT(*) AS num_reviews
    FROM public.reviews_clean r
    JOIN public.apartamentos_clean a ON r.listing_id = a.id
    WHERE 
        EXTRACT(MONTH FROM r.date) = 3 -- Marzo (mes de Fallas)
        AND a.review_scores_rating >= 4
    GROUP BY a.neighbourhood_cleansed, EXTRACT(YEAR FROM r.date)
    HAVING COUNT(*) > 30
),
crecimiento AS (
    SELECT
        barrio,
        anio,
        num_reviews,
        LAG(num_reviews) OVER (PARTITION BY barrio ORDER BY anio) AS prev_year_reviews,
        num_reviews - LAG(num_reviews) OVER (PARTITION BY barrio ORDER BY anio) AS crecimiento
    FROM fallas_reviews
)
SELECT *
FROM crecimiento
WHERE crecimiento IS NOT NULL
ORDER BY crecimiento DESC
LIMIT 1;

```
El gráfico que he usado para visualizar esta información es un gráfico de líneas múltiple, porque permite comparar de manera clara y visual la evolución interanual del número de reseñas durante el periodo de Fallas en los diferentes barrios, facilitando la identificación de qué barrio ha experimentado el mayor crecimiento y mostrando las tendencias a lo largo del tiempo para cada zona.
</div>

---

<a id="section5"></a>
# <font color="#00586D"> 5. Paneles e informes</font>

En este punto, ya hemos creado una conexión a la base de datos, hemos creado los datasets y hemos lanzado algunas consultas SQL para obtener información básica de los datos, a partir de la cual también hemos generado gráficos concretos. El último caso que nos queda por cubrir es la creación de paneles e informes. En concreto, vamos a tener en cuenta dos escenarios diferentes



<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicio 5</b></font><br>

1. Imaginemos que somos un nuevo inversor que quiere estudiar el panorama de los alquileres turísticos en Valencia. Para ello, vamos a crear un panel que contenga la información más relevante para un inversor. En este caso, el panel debería contener al menos 5 gráficos diferentes y un informe que explique la información que contiene cada gráfico y por qué es relevante para un inversor. Qué zonas son más rentables, cuáles han experimentado un mayor crecimiento, cuáles son los mejores barrios para invertir, etc.

📌 KPIs clave:

- Ingreso estimado anual (precio × nº de reseñas)
- Ocupación estimada (proxy por frecuencia de reviews)
- Puntuación media del barrio
- Crecimiento de demanda interanual (reseñas o precios)

❓ Preguntas típicas:
- ¿Qué barrios tienen mayor ingreso por alojamiento?
- ¿Dónde están creciendo más las reseñas positivas?
- ¿Cuáles son los tipos de alojamiento más rentables?
- ¿Cómo ha cambiado la oferta en zonas clave desde 2020?

📈 Visualizaciones recomendadas:
- Mapa de ingresos por barrio
- Serie temporal de reviews y precio medio
- Scatter: rating vs nº de reseñas por barrio


2. Imaginemos que somos un gran propietario que ya dispone más de 10 propiedades en Valencia y que quiere estudiar el contexto de sus propiedades y de las zonas, para poder hacer asi inversiones para mejorar los alojamientos, etc. 

📌 KPIs clave:
- Nº de reviews por propiedad
- Puntuación media por propiedad
- Variación mensual
- Distribución por tipo de habitación y disponibilidad
- Alertas de caída en reseñas o puntuación

❓ Preguntas típicas:
- ¿Qué apartamentos tienen puntuaciones descendentes?
- ¿Qué propiedades dejaron de recibir reviews?
- ¿Cuáles reciben menos de X reseñas por mes?
- ¿Hay propiedades con alto precio pero baja reputación?

📈 Visualizaciones recomendadas:
- Tabla detallada de listings con filtros por host
- Panel de control por propiedad (reviews, rating, precio)
- Gráfico de dispersión: precio vs. puntuación

Una vez creados los dashboards, exportalos a pdf y añadelos dentro de la carpeta `resultados` que acompañará al entregable de este capstone. 

---

<a id="section6"></a>
# <font color="#00586D"> 6. Casos de negocio</font>

Mas allá de los casos que hemos presentado para realizar el análisis, plantea 3 escenarios analíticos en los que tendría sentido por ejemplo cruzar con otros datos externos para realizar análisis más completos o complejos y como crees que las funcionalidades de Preset/Superset podrían cubrirlos.

<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>

**Caso de uso 1**

El primer caso de uso que plantearía es analizar el impacto de la inflación en precios y demanda, que consistiría en cruzar precios y reviews con indicadores como el IPC o la tasa de paro.
Los principales conjuntos de datos que intervendrían son apartamentos_clean, reviews_clean y datos macroeconómicos externos.
Las métricas serían evolución de precios y reviews frente al IPC, porque permiten ver cómo afecta el contexto económico.
Dentro de Preset, vendría bien por sus filtros temporales y capacidad para combinar series de datos.

**Caso de uso 2**

El segundo caso de uso que plantearía es comparar Airbnb con hoteles tradicionales, que consistiría en cruzar datos de apartamentos con precios, estrellas y ubicación de hoteles.
Los conjuntos de datos serían apartamentos_clean, reviews_clean y una fuente externa de hoteles.
Usaría mapas de precios, comparativas de puntuación y scatterplots, porque muestran diferencias de valor por zona.
Preset permite esto fácilmente con filtros, mapas interactivos y métricas personalizadas.

**Caso de uso 3**

El tercer caso de uso que plantearía es detectar zonas emergentes para inversión turística, que consistiría en cruzar datos de actividad en Airbnb con licencias urbanas y accesibilidad.
Se usarían apartamentos_clean, reviews_clean y fuentes externas de urbanismo y transporte.
Las métricas serían crecimiento de reviews, baja saturación y mejoras en infraestructuras, porque identifican oportunidades.
Preset facilita este análisis con mapas, filtros por barrio y gráficos de evolución.


<div style="
    background-color: #2E8B57; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Ejemplo</h2>

El primer caso de uso que plantearía es analizar el impacto de eventos culturales en la actividad de Airbnb, que consistiría en cruzar los datos de reviews y precios con calendarios de eventos locales (como Fallas, Mobile World Congress, San Fermín, etc.). Los principales conjuntos de datos que intervendrían son reviews, apartamentos y una fuente externa con fechas de eventos relevantes por ciudad. Las principales métricas o gráficos que utilizaría son número de reviews por mes, variación de precio medio, ratio de reservas por barrio antes, durante y después de eventos porque permite entender el impacto económico y operativo que estos eventos generan en la oferta de corta estancia.
Dentro de las funcionalidades de Preset, creo que vendría bien porque permite usar filtros temporales, desglosar por dimensión geográfica y generar gráficos de evolución comparando periodos o categorías (por ejemplo, marzo vs. resto del año, barrios turísticos vs. residenciales).

<div style="
    background-color: #00586D; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Comentarios ejercicio</h2>
Mas allá de la parte de negocio, intenta plantear estas preguntas:

**1.	¿Qué barreras técnicas o de calidad de datos podrían surgir al incorporar múltiples ciudades al análisis?**

Inconsistencias en los nombres de barrios, formatos de fechas y monedas, así como disponibilidad desigual de reviews o precios. También puede haber diferencias en la granularidad geográfica o faltantes en campos clave.

**2.	¿Qué ciudades serían prioritarias para ampliar el estudio y por qué?**

Barcelona, Madrid, Sevilla y Málaga, por su volumen turístico, eventos recurrentes y alta densidad de oferta en Airbnb. Son ciudades con suficiente volumen de datos para análisis comparables y relevantes.

**3.	¿Qué diferencias estructurales existen entre ciudades turísticas y no turísticas en cuanto a oferta, precio o reputación?**

Las ciudades turísticas suelen tener mayor concentración de alojamientos, precios más estacionales y puntuaciones más polarizadas. Las no turísticas muestran menos rotación y una oferta más homogénea.

**4.	¿Cómo evoluciona la demanda (reseñas) en los meses clave de cada ciudad?**

Tiende a haber picos marcados en verano y eventos locales, como Fallas o Semana Santa. La estacionalidad varía según el clima y la popularidad internacional de cada ciudad.

**5.	¿Qué perfiles de alojamiento funcionan mejor en cada contexto urbano?**

En zonas céntricas destacan los estudios y apartamentos pequeños. En ciudades más familiares o con turismo nacional, funcionan mejor viviendas completas con varias habitaciones.

**6.	¿Qué patrones comunes o divergentes emergen al comparar barrios equivalentes entre ciudades distintas?**

Barrios céntricos suelen compartir alta demanda y precios elevados, pero difieren en puntuaciones y rotación. Barrios periféricos varían mucho en reputación y rentabilidad según la ciudad.

**7. ¿Cómo podríamos segmentar a los huéspedes por nacionalidad o perfil cultural analizando sus comentarios con modelos de lenguaje?**

Usando NLP para detectar idioma, referencias culturales, preferencias y sentimientos. Modelos multilingües pueden ayudar a inferir nacionalidad y clasificar patrones de comportamiento o expectativas.


<div style="
    background-color: #2E8B57; 
    color: white; 
    padding: 20px; 
    font-size: 12px;
    border-radius: 14px;
    font-family: Arial, sans-serif;
">
    <h2 style="margin-bottom: 5px;"> Ejemplo</h2>

**8. ¿Qué valor añadido aportaría cruzar los datos de alojamiento con variables externas como clima, precios hoteleros o eventos urbanos relevantes?**

Cruzar los datos de Airbnb con fuentes externas como información meteorológica, precios de hoteles tradicionales o calendario de eventos culturales permitiría contextualizar los cambios en la demanda o el precio más allá del comportamiento interno de la plataforma. Por ejemplo, podríamos explicar por qué ciertos barrios suben de precio puntualmente (evento masivo) o por qué bajan las reseñas en ciertos meses (mal clima). Además, integrar estas variables facilitaría construir modelos predictivos más robustos y ofrecer análisis más cercanos a una visión 360 del mercado turístico. En Preset, esto puede resolverse fácilmente mediante datasets combinados (table joins o custom SQL), filtros dinámicos por periodo y visualizaciones comparativas entre dimensiones internas y externas.